## base fine-tuning

To make experimental pipeline, doing simply fine-tuning
after that, saved to local env


required: colab-pro environment

In [1]:
!pip install -r https://raw.githubusercontent.com/fumiya2001/culminating_project/refs/heads/main/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 137.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 59.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 56.1 MB/s eta 0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556704 sha256=6c5f0854133a1c7496d8c00d630e9dab41a14e71eaf81b025dabdb5fb885f281
  Stored in directory: /root/.cache/pip/wheels/93/8b/55/dd5978a069678c372520847cf84ba2ec539cb41917c00a2206
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=19e72f99255eb90c66c4500b31db75792dfce02bc93e2722d316e7446e694321
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built ipadic unidic-li

In [18]:
!nvidia-smi

Wed Mar 25 22:18:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             30W /   70W |    3255MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import  torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [5]:
# check the model size and vocab size
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Model size: {model.num_parameters() / 1e9:.2f}B parameters")

Tokenizer vocab size: 151643
Model size: 1.54B parameters


In [6]:
# model anatomy
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [7]:
# chenck the first layer's self-attention q_proj weight
model.model.layers[0].self_attn.q_proj.weight

Parameter containing:
tensor([[ 0.0131, -0.0091,  0.0015,  ...,  0.0093,  0.0023,  0.0109],
        [ 0.0009, -0.0022, -0.0194,  ..., -0.0189,  0.0047, -0.0079],
        [ 0.0048,  0.0188, -0.0040,  ..., -0.0102,  0.0092,  0.0118],
        ...,
        [-0.0057, -0.0151, -0.0003,  ...,  0.0016, -0.0060, -0.0308],
        [-0.0063,  0.0137, -0.0100,  ...,  0.0225, -0.0044, -0.0090],
        [ 0.0088, -0.0203,  0.0071,  ...,  0.0178,  0.0071,  0.0157]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)

reference: https://huggingface.co/datasets/rajpurkar/squad_v2

In [8]:
from datasets import load_dataset

# question and answer parir dataset
dataset = load_dataset('squad_v2')

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [10]:
train_df = dataset['train']
test_df = dataset['validation']

train_df = train_df.filter(lambda x: len(x["answers"]["text"]) > 0)
test_df = test_df.filter(lambda x: len(x["answers"]["text"]) > 0)

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [37]:
# check the answer length distribution
dataset['train'][0]

{'id': '56be85543aeaaa14008c9063',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': 'When did Beyonce start becoming popular?',
 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}

In [39]:
dataset['train'][2]

{'id': '56be85543aeaaa14008c9066',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': "When did Beyonce leave Destiny's Child and become a solo singer?",
 'answers': {'text': ['2003'], 'answer_start': [526]}}

In [12]:
def preprocess_function(example):
    answer_text = example["answers"]["text"][0]

    prompt = (
        f"### Context:\n{example['context']}\n\n"
        f"### Question:\n{example['question']}\n\n"
        "### Answer:\n"
    )
    answer = answer_text + tokenizer.eos_token
    full_text = prompt + answer

    full_tokens = tokenizer(full_text, truncation=True, max_length=768)
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=768)

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = input_ids.copy()
    prompt_len = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [13]:
tokenized_dataset = train_df.map(preprocess_function, batched=False)

Map:   0%|          | 0/86821 [00:00<?, ? examples/s]

In [15]:
tokenized_dataset_test = test_df.map(preprocess_function, batched=False)

Map:   0%|          | 0/5928 [00:00<?, ? examples/s]

In [16]:
tokenized_dataset.column_names

['id',
 'title',
 'context',
 'question',
 'answers',
 'input_ids',
 'attention_mask',
 'labels']

In [17]:
tokenized_dataset = tokenized_dataset.select_columns(["input_ids", "attention_mask", "labels"])
tokenized_dataset_test = tokenized_dataset_test.select_columns(["input_ids", "attention_mask", "labels"])

In [19]:
# select a sample from the tokenized dataset
sample = tokenized_dataset.shuffle(seed=42).select(range(2000))
sample_test = tokenized_dataset_test.shuffle(seed=42).select(range(1000))

In [21]:
from transformers import (
    Trainer, 
    TrainingArguments,
    DataCollatorForSeq2Seq
)

data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, return_tensors="pt")

training_args = TrainingArguments(
    output_dir="./outputs/full_finetune_3/25",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=20,
    save_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    save_total_limit=2,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sample,
    eval_dataset=sample_test,
    data_collator=data_collator,
)


In [22]:
trainer.train()

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.3016781539916992, metrics={'train_runtime': 2346.5338, 'train_samples_per_second': 0.852, 'train_steps_per_second': 0.053, 'total_flos': 3021839372457984.0, 'train_loss': 0.3016781539916992, 'epoch': 1.0})

In [27]:
trainer.save_model("./outputs/full_finetune_3_25")
tokenizer.save_pretrained("./outputs/full_finetune_3_25")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./outputs/full_finetune_3_25/tokenizer_config.json',
 './outputs/full_finetune_3_25/chat_template.jinja',
 './outputs/full_finetune_3_25/tokenizer.json')

In [28]:
import os
print(os.getcwd())

/content


In [30]:
!zip -r model.zip outputs/full_finetune_3_25

  adding: outputs/full_finetune_3_25/ (stored 0%)
  adding: outputs/full_finetune_3_25/config.json (deflated 71%)
  adding: outputs/full_finetune_3_25/tokenizer.json (deflated 81%)
  adding: outputs/full_finetune_3_25/generation_config.json (deflated 28%)
  adding: outputs/full_finetune_3_25/chat_template.jinja (deflated 71%)
  adding: outputs/full_finetune_3_25/model.safetensors (deflated 21%)
  adding: outputs/full_finetune_3_25/tokenizer_config.json (deflated 60%)
  adding: outputs/full_finetune_3_25/training_args.bin (deflated 53%)


In [32]:
from google.colab import files
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [36]:
!cp model.zip /content/drive/MyDrive/